In [1]:
import gzip
import numpy as np
import pandas as pd


In [3]:
# HLA-DQA1 is not listed because it consistently didn't pass QC in all datasets using HLA-HD
classical_class_I_II = {'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DQB1', 'HLA-DRA', 'HLA-DRB1'}

# MACs
mac_thresholds = [5, 10, 20, 50]

In [4]:
nocov_tsv = {
    'ALL': '../data/1000G/GWAS_continuous/ALL/seed_20_nocov.tsv.gz',
    'AFR': '../data/1000G/GWAS_continuous/AFR/seed_21_nocov.tsv.gz',
    'AMR': '../data/1000G/GWAS_continuous/AMR/seed_25_nocov.tsv.gz',
    'EAS': '../data/1000G/GWAS_continuous/EAS/seed_22_nocov.tsv.gz',
    'EUR': '../data/1000G/GWAS_continuous/EUR/seed_23_nocov.tsv.gz',
    'SAS': '../data/1000G/GWAS_continuous/SAS/seed_24_nocov.tsv.gz'
}

cov_tsv = {
    'ALL': '../data/1000G/GWAS_continuous/ALL/seed_20_cov.tsv.gz',
    'AFR': '../data/1000G/GWAS_continuous/AFR/seed_21_cov.tsv.gz',
    'AMR': '../data/1000G/GWAS_continuous/AMR/seed_25_cov.tsv.gz',
    'EAS': '../data/1000G/GWAS_continuous/EAS/seed_22_cov.tsv.gz',
    'EUR': '../data/1000G/GWAS_continuous/EUR/seed_23_cov.tsv.gz',
    'SAS': '../data/1000G/GWAS_continuous/SAS/seed_24_cov.tsv.gz'
}

In [5]:
def read_pvalue_tsv(filepath, mac):
    df = pd.read_csv(filepath, sep = "\t")
    df['gene'] = df['allele'].apply(lambda x: x.split('*')[0])

    assert len(df[~df.error.isna()]) == 0 # Any convergence errors? 
    
    df = df[df.gene.isin(classical_class_I_II)] # Keep only classical class I and II
        
    df_temp = df[df.mac >= mac]

    # Get all unique alleles and make sure that they all had the same AC values
    df_alleles = df_temp.groupby('allele')[['ac']].nunique().reset_index()
    assert len(df_alleles[df_alleles.ac != 1]) == 0 
    n_alleles = len(df_alleles)

    # Get single minimal p-value per simulation
    df_pvalues = df_temp[["output", "pheno", "p"]].groupby(["output", "pheno"]).min().reset_index()
    n_simulations = len(df_pvalues)
    min_pvalues_q5 = np.quantile(df_pvalues.p, 0.05)
    m_eff = 0.05 / min_pvalues_q5
        
    return n_simulations, n_alleles, min_pvalues_q5, m_eff



In [6]:
print("With covariates")

print(f'Dataset;', end = "")
for mac in mac_thresholds:
    print(f';N_alleles [MAC>={mac}]; M_eff [MAC>={mac}]', end = "")
print()

for label in ['ALL', 'AFR', 'AMR', 'EAS', 'EUR', 'SAS']:
    print(f'{label}', end = "")
    for mac in mac_thresholds:
        n_simulations, n_alleles, min_pvalues_q5, m_eff = read_pvalue_tsv(cov_tsv[label], mac)
        print(f';{n_alleles};{m_eff:.0f} ({m_eff / n_alleles * 100:.0f})', end = "")
    print()
    

With covariates
Dataset;;N_alleles [MAC>=5]; M_eff [MAC>=5];N_alleles [MAC>=10]; M_eff [MAC>=10];N_alleles [MAC>=20]; M_eff [MAC>=20];N_alleles [MAC>=50]; M_eff [MAC>=50]
ALL;283;260 (92);236;216 (91);196;182 (93);143;130 (91)
AFR;148;134 (91);128;116 (90);103;93 (90);62;54 (88)
AMR;153;142 (93);109;104 (96);64;58 (90);21;20 (95)
EAS;135;124 (92);110;100 (91);84;75 (89);44;39 (89)
EUR;133;117 (88);105;93 (89);77;66 (86);43;36 (85)
SAS;138;128 (92);107;103 (96);77;70 (91);46;43 (93)


In [7]:
print("Without covariates")

print(f'Dataset;', end = "")
for mac in mac_thresholds:
    print(f';N_alleles [MAC>={mac}]; M_eff [MAC>={mac}]', end = "")
print()

for label in ['ALL', 'AFR', 'AMR', 'EAS', 'EUR', 'SAS']:
    print(f'{label}', end = "")
    for mac in mac_thresholds:
        n_simulations, n_alleles, min_pvalues_q5, m_eff = read_pvalue_tsv(nocov_tsv[label], mac)
        print(f';{n_alleles};{m_eff:.0f} ({m_eff / n_alleles * 100:.0f})', end = "")
    print()

Without covariates
Dataset;;N_alleles [MAC>=5]; M_eff [MAC>=5];N_alleles [MAC>=10]; M_eff [MAC>=10];N_alleles [MAC>=20]; M_eff [MAC>=20];N_alleles [MAC>=50]; M_eff [MAC>=50]
ALL;283;281 (99);236;235 (100);196;193 (98);143;136 (95)
AFR;148;130 (88);128;113 (88);103;93 (90);62;55 (88)
AMR;153;142 (93);109;103 (95);64;59 (91);21;20 (94)
EAS;135;123 (91);110;98 (89);84;75 (89);44;39 (90)
EUR;133;117 (88);105;94 (90);77;68 (89);43;37 (85)
SAS;138;131 (95);107;101 (94);77;72 (93);46;42 (91)
